# 第 5 课 · 超时、重试与编码

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课解决两个真实痛点：**网络偶尔卡死 / 失败怎么办**，以及**抓到的中文是乱码怎么办**。
> 只引入三个新概念：**timeout、重试、.text 与 .content 的编码差异**。

In [1]:
# 每个 notebook 开头都要重新 import 这一课要用的库
import requests                                   # HTTP 请求库（每本 notebook 都要重新 import）
print('requests 版本：', requests.__version__)

requests 版本： 2.34.2


## 5.1 为什么一定要写 `timeout=`

**类比**：你给同学打电话，对方一直不接。如果不设“最多等几秒”，你就会永远举着手机站在那里。

**事实**：`requests.get(url)` **默认是没有超时的**——如果服务器不响应，你的程序会永远卡在那一行，看起来像死机。

**做法**：永远加上 `timeout=秒数`。下面故意请求一个“服务器要 5 秒才响应”的地址，却只允许等 1 秒，看看会发生什么：

In [2]:
try:
    # httpbin 的 /delay/5 表示服务器故意拖延 5 秒再回答
    resp = requests.get('https://httpbin.org/delay/5', timeout=1)
    print('意外：居然成功了？', resp.status_code)
except requests.exceptions.Timeout as e:
    # 这是我们预期会走到的分支
    print('符合预期：1 秒内没等到，抛出 Timeout，程序没有傻等。')
except Exception as e:
    print('其它网络问题（可能没联网）：', type(e).__name__, e)

符合预期：1 秒内没等到，抛出 Timeout，程序没有傻等。


## 5.2 手写一个最简单的重试循环

网络偶尔抽风（超时、连接被重置）是常态。工程上的做法：**失败就重试几次，连续失败才算真失败**。

这里**只用一个 for 循环**，不引入 `urllib3.Retry` 那种复杂配置，先看懂原理：

In [3]:
def fetch_with_retry(url, times=3, timeout=10):
    """最多请求 times 次；成功就立刻返回响应，全部失败才返回 None。"""
    for i in range(1, times + 1):          # i 从 1 数到 3
        try:
            resp = requests.get(url, timeout=timeout)
            print(f'第 {i} 次请求成功，状态码', resp.status_code)
            return resp                     # 成功就直接返回，不再重试
        except Exception as e:
            print(f'第 {i} 次失败：', type(e).__name__, e)
    print('三次都失败了，放弃。')
    return None


result = fetch_with_retry('https://httpbin.org/get')   # 换成你自己想请求的网址也可以

第 1 次请求成功，状态码 200


## 5.3 乱码的根源：`.content` 是字节，`.text` 是解码后的字符串

抓到中文网页却看到 `鍦ㄧ殑...` 这种鬼画符，几乎都是**编码问题**。先把概念理清：

- **`response.content`**：服务器发回来的**原始字节（bytes）**，还没被翻译成文字。
- **`response.text`**：requests 猜测一个编码，把上面的字节**解码成字符串（str）**给你。

问题在于：requests **经常猜错编码**。同一个字节串，用错编码解码就会乱码。

下面这段**完全离线**，亲手体会“字节是同一份，编码选错就乱码”：

In [4]:
raw = '你好，爬虫同学'.encode('utf-8')     # 中文先按 utf-8 编成字节
print('原始字节 bytes：', raw)
print('用 utf-8 正确解码：', raw.decode('utf-8'))
try:
    print('用 gbk 错误解码：', raw.decode('gbk'))   # 多半变成乱码或直接报错
except Exception as e:
    print('gbk 解码失败：', type(e).__name__, e)

原始字节 bytes： b'\xe4\xbd\xa0\xe5\xa5\xbd\xef\xbc\x8c\xe7\x88\xac\xe8\x99\xab\xe5\x90\x8c\xe5\xad\xa6'
用 utf-8 正确解码： 你好，爬虫同学
gbk 解码失败： UnicodeDecodeError 'gbk' codec can't decode byte 0xab in position 14: illegal multibyte sequence


## 5.4 怎么修：`response.encoding` 与 `apparent_encoding`

- **`response.encoding`**：requests 当前用来解码 `.text` 的编码（它自己猜的，可能猜错）。
- **`response.apparent_encoding`**：requests 用 `chardet` 库**真正去分析字节内容**后猜的编码，通常更准。

**修复套路**：先打印这两个值对比，再手动把 `response.encoding` 改成更可靠的那个，然后再看 `.text`。

下面用豆瓣中文页验证（联网用 try/except，断网也不报错）：

In [5]:
browser_ua = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'

try:
    r = requests.get('https://movie.douban.com/top250',
                     headers={'User-Agent': browser_ua}, timeout=10)
    print('requests 猜的 encoding：', r.encoding)
    print('chardet 分析出的 apparent_encoding：', r.apparent_encoding)

    # 手动纠正编码，再取文本
    r.encoding = r.apparent_encoding
    print('纠正编码后前 100 个字：')
    print(r.text[:100])
except Exception as e:
    print('这次没联网，不影响学习：', type(e).__name__, e)

requests 猜的 encoding： utf-8
chardet 分析出的 apparent_encoding： utf-8
纠正编码后前 100 个字：
<!DOCTYPE html>
<html lang="zh-CN" class="ua-windows ua-webkit">
<head>
    <meta http-equiv="Conten


## 5.5 小结（记住这四句）

1. **永远写 `timeout=`**，否则服务器不响应时程序会永远卡住。
2. **重试就是一个 for 循环**：最多 N 次，成功就 `return`，失败记下来继续。
3. `.content` 是**原始字节**，`.text` 是按编码解码后的**字符串**。
4. 中文乱码：对比 `response.encoding` 和 `response.apparent_encoding`，手动改对编码再看 `.text`。

## 5.6 随堂小练习（自己动手，空格里写）

1. 给 5.2 的 `fetch_with_retry` 增加一个参数 `sleep_seconds`，每次失败后 `time.sleep(sleep_seconds)` 再重试（记得 `import time`）。
2. 找一个你本地的中文文本文件，读成 bytes，分别用 utf-8 和 gbk 解码，对比打印结果。

In [6]:
# TODO：自己写代码完成上面两题（这是随堂小练，没有单独答案文件）


## 来源与延伸

- requests 官方文档（超时）：https://requests.readthedocs.io/en/latest/user/quickstart/#timeouts
- requests 官方文档（编码）：https://requests.readthedocs.io/en/latest/user/quickstart/#response-content
- MDN 字符编码介绍：https://developer.mozilla.org/zh-CN/docs/Glossary/Character_encoding
- httpbin delay 接口：https://httpbin.org/#/Dynamic_data/get_delay__delay_
- 下一阶段：拿到干净文本后，怎么从 HTML 树里把数据抠出来（BeautifulSoup / XPath）。